<a href="https://colab.research.google.com/github/Gayatri2002/MBTI-personality-prediction-using-NLP/blob/main/tarp_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Complete Chat Analysis And Personality
Detection Using NLP**   TARP Project


In [ ]:
pip install anvil-uplink

In [ ]:
pip install catboost

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn import metrics
from sklearn.metrics import classification_report, f1_score
import pickle
import os.path
import plotly.offline as pyo
import plotly.graph_objs as go
import spacy
from nltk.stem.snowball import SnowballStemmer
from catboost import CatBoostClassifier
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MBTI 500.csv')

In [ ]:
df.head()

In [ ]:
df['posts'][0]

In [ ]:
df['type'][0]

In [ ]:
df['type'].unique()
TYPES=['ENFJ','ENFP','ENTJ','ENTP','ESFJ','ESFP','ESTJ','ESTP','INFJ','INFP','INTJ','INTP','ISFJ','ISFP','ISTJ','ISTP']

In [ ]:
print(f"Total of {len(df['type'].unique())} types of classified MBTI posts")

In [ ]:
df.isnull().sum()

Data Visualization

In [ ]:
df_bar_chart=df.groupby('type').count()


trace1 = go.Bar(x=df_bar_chart.index, y=df_bar_chart['posts'])

data = [trace1]
layout = go.Layout(title='MBTI # Classified Posts per Type')

fig = go.Figure(data=data, layout=layout)

fig.show()

In [ ]:
filename = '/content/drive/MyDrive/Colab Notebooks/mbti_svm_v1.sav'

In [ ]:
recreate_model=True

In [ ]:
X = df['posts'] # features
y = df['type']  # labels
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Check if need to recreate the model
if recreate_model:    
    
    # Creating an instance to vectorizer:
    vectorizer = TfidfVectorizer()
    
    # Training the vectorizer:
    X_train_tfidf = vectorizer.fit_transform(X_train)
    
    # Training the classifier:
    clf = LinearSVC()
    clf.fit(X_train_tfidf, y_train)
    
    # Pipelining the vectorizer and the classifier
    text_clf = Pipeline([('tfidf',TfidfVectorizer()),('clf',LinearSVC())])
    text_clf.fit(X_train, y_train)
    
    # saving the model to disk
    pickle.dump(text_clf, open(filename, 'wb'))

# If there is no need to recreate the model, just open the file from the disk    
else:
    # loading the model from disk
    text_clf = pickle.load(open(filename, 'rb'))

In [ ]:
predictions = text_clf.predict(X_test)

Classification report for the SVC Linear model

In [ ]:
print(classification_report(y_test, predictions))

Accuracy for the SVC Linear model

In [ ]:
print(f"Overall accuracy of the model: {round(metrics.accuracy_score(y_test, predictions),2)}")

Example execution for SVC Linear model

In [ ]:
str(text_clf.predict(["Hello world"])[0])

In [ ]:
import nltk
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

class Lemmatizer(object):
    def __init__(self):
        self.lemmatizer = WordNetLemmatizer()
    def __call__(self, sentence):
        return [self.lemmatizer.lemmatize(word) for word in sentence.split() if len(word)>2]

In [ ]:
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import RegexpTokenizer
import pickle
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

filename2 = '/content/drive/MyDrive/Colab Notebooks/mbti_svm_v1.sav'
model_cat = pickle.load(open(filename2, 'rb'))

target_encoder = LabelEncoder()

tokenizer = RegexpTokenizer(r'\w+')

vectorizer = TfidfVectorizer(
    max_features=5000,
    stop_words='english',
    tokenizer=lambda text: [WordNetLemmatizer().lemmatize(token.lower()) for token in tokenizer.tokenize(text)],
    lowercase=True
)

vectorizer.fit(X_train)

try:
    X_test = vectorizer.transform(X_test).toarray()
    y_test = target_encoder.fit_transform(y_test)
except Exception as e:
    print(f"Error during vectorization: {e}")
    print(f"Problematic text: {X_test}")




**Catboost model**

In [ ]:
pip install catboost


In [ ]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from wordcloud import WordCloud
from tqdm import tqdm
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
import re
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC,LinearSVC
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from xgboost import XGBClassifier
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.experimental import enable_hist_gradient_boosting
from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.over_sampling import SMOTE
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')
from google.colab import drive
drive.mount('/content/drive')
data=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/MBTI 500.csv')
data.head()

In [ ]:
train_data,test_data=train_test_split(data,test_size=0.2,random_state=42,stratify=data.type)

In [ ]:
def clear_text(data):
    data_length=[]
    lemmatizer=WordNetLemmatizer()
    cleaned_text=[]
    for sentence in tqdm(data.posts):
        sentence=sentence.lower()
        
#         removing links from text data
        sentence=re.sub('https?://[^\s<>"]+|www\.[^\s<>"]+',' ',sentence)
    
#         removing other symbols
        sentence=re.sub('[^0-9a-z]',' ',sentence)
    
        
        data_length.append(len(sentence.split()))
        cleaned_text.append(sentence)
    return cleaned_text,data_length

In [ ]:
train_data.posts,train_length=clear_text(train_data)

In [ ]:
test_data.posts,test_length=clear_text(test_data)

In [ ]:
import nltk
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import TfidfVectorizer


lemmatizer = WordNetLemmatizer()
vectorizer = TfidfVectorizer(max_features=5000, stop_words='english', tokenizer=lemmatizer.lemmatize)
vectorizer.fit(train_data.posts)

In [ ]:
train_post=vectorizer.transform(train_data.posts).toarray()
test_post=vectorizer.transform(test_data.posts).toarray()

In [ ]:
train_post.shape

In [ ]:
target_encoder=LabelEncoder()
train_target=target_encoder.fit_transform(train_data.type)
test_target=target_encoder.fit_transform(test_data.type)

In [ ]:

from catboost import CatBoostClassifier

model_cat = CatBoostClassifier(task_type="CPU")
model_cat.fit(train_post, train_target)


In [ ]:
print(accuracy_score(y_test,model_cat.predict(X_test)))

For connecting to Anvil

In [ ]:
import anvil.server

anvil.server.connect("YFVBMK7CHYRL4A2NRUCXDR3S-4QODO5CLYDAYIUP6")

In [ ]:
@anvil.server.callable
def SVC(txt):
    print(txt)
    type=text_clf.predict([txt])
    return str(type[0])
@anvil.server.callable
def CAT(txt):
  text=vectorizer.transform([txt]).toarray()
  ans=model_cat.predict(text)
  return str(TYPES[ans[0][0]])